In [1]:
import math
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans
import numpy as numpy

In [2]:
Data = pd.read_csv("bank_survey.csv")
cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
data_encode = Data.drop(cols, axis=1)
data_encode = data_encode.apply(LabelEncoder().fit_transform)
data_rest = Data[cols]
Data = pd.concat([data_rest, data_encode], axis=1)
print(Data.head())

   age  balance  day  duration  campaign  pdays  previous  job  marital  \
0   58     2143    5       261         1     -1         0    4        1   
1   44       29    5       151         1     -1         0    9        2   
2   33        2    5        76         1     -1         0    2        1   
3   47     1506    5        92         1     -1         0    1        1   
4   33        1    5       198         1     -1         0   11        2   

   education  default  housing  loan  contact  month  poutcome  y  
0          2        0        1     0        2      8         3  0  
1          1        0        1     0        2      8         3  0  
2          1        0        1     1        2      8         3  0  
3          3        0        1     0        2      8         3  0  
4          3        0        0     0        2      8         3  0  


In [3]:
data_train,data_test = train_test_split(Data, test_size=0.5, random_state=4)
x_train=data_train.drop("y",axis=1)
y_train=data_train["y"]
x_test=data_test.drop("y",axis=1)
y_test=data_test["y"]

scaler=StandardScaler()
scaler.fit(x_train)
x_train=scaler.fit_transform(x_train)
x_test=scaler.fit_transform(x_test)

In [4]:
K_cent=8
km=KMeans(n_clusters=K_cent,max_iter=99)
km.fit(x_train)
cent=km.cluster_centers_

In [5]:
max=0
for i in range(K_cent):
  for j in range(K_cent):
    d=numpy.linalg.norm(cent[i]-cent[j])
    if(d>max):
      max=d
d=max
sigma=d/math.sqrt(2*K_cent)
print(sigma)

2.3219944601824873


In [6]:
shape=x_train.shape
row=shape[0]
column=K_cent
G=numpy.empty((row,column),dtype=float)
for i in range(row):
  for j in range(column):
    dist=numpy.linalg.norm(x_train[i]-cent[j])
    G[i][j]=math.exp(-math.pow(dist,2)/math.pow(2*sigma,2))
print(G)

[[0.07696666 0.26312287 0.66718403 ... 0.23050994 0.21075729 0.02061425]
 [0.21213285 0.58825448 0.31462334 ... 0.78996557 0.44486332 0.0506799 ]
 [0.30533081 0.71186917 0.36176132 ... 0.57652985 0.50453031 0.04769539]
 ...
 [0.12611798 0.52840225 0.24946676 ... 0.33187785 0.41050352 0.04099426]
 [0.16303095 0.55389606 0.29330214 ... 0.54722772 0.68140392 0.04525561]
 [0.22557888 0.60248571 0.32902455 ... 0.71817593 0.48599553 0.05149061]]


In [7]:
GTG=numpy.dot(G.T,G)
GTG_inv=numpy.linalg.inv(GTG)
fac=numpy.dot(GTG_inv,G.T)
w=numpy.dot(fac,y_train)
print(w)

[ 0.16576281 -0.58248996  0.25860178  1.30096387  0.08193107 -0.03584314
 -0.38971575  0.08951254]


In [8]:
row=x_test.shape[0]
column=K_cent
G_test=numpy.empty((row,column),dtype=float)
for i in range(row):
  for j in range(column):
    dist=numpy.linalg.norm(x_test[i]-cent[j])
    G_test[i][j]=math.exp(-math.pow(dist,2)/math.pow(2*sigma,2))
print(G_test[0])

[0.17296243 0.60217894 0.30354755 0.33024081 0.62543106 0.76297488
 0.46074175 0.05021783]


In [9]:
prediction=numpy.dot(G_test,w)
prediction=0.5*(numpy.sign(prediction-0.5)+1)
score=accuracy_score(y_test,prediction)
print(score)

0.8893214190922764
